In [4]:
import pandas as pd
import numpy as np

activities = pd.read_csv(
    "../data/raw/chembl_leishmania_donovani_activities.csv",
    sep=";",
    engine="python",
    on_bad_lines="skip"
)

print(activities.shape)

(12257, 48)


This created activities_nm which contains compounds having their Standard Units as nM only.

In [5]:
activities_nm=activities[activities["Standard Units"]=="nM"].copy()

print(activities_nm.shape)

(10296, 48)


We checked for how many compounds have null Standard value

In [6]:
activities_nm["Standard Value"].isnull().sum()

np.int64(39)

The compounds having null Standard values were deleted.

In [7]:
activities_nm=activities_nm.dropna(subset=["Standard Value"])
print(activities_nm.shape)

(10257, 48)


It was verified if the compounds having null Standard value have been deleted.

In [8]:
activities_nm["Standard Value"].isnull().sum()

np.int64(0)

The pIC50 values were calculated using the formula:
pIC50 = 9 - log10(IC50 [nM])

This converts the IC50 value in nanomolar units into a logarithmic potency score.

In [20]:
activities_nm["pIC50"] = (
    9 - np.log10(activities_nm["Standard Value"])
)
activities_nm["pIC50"].describe()

count    10248.000000
mean         4.955206
std          0.859658
min          2.000000
25%          4.300000
50%          4.832683
75%          5.422796
max          9.920819
Name: pIC50, dtype: float64

## Checking for Invalid Activity Values

The dataset was examined for compounds with IC50 values less than or equal to zero. Such values are invalid for pIC50 calculations because logarithmic transformations cannot be applied to zero or negative values.

In [ ]:
activities_nm[
    activities_nm["Standard Value"] <= 0
][
    [
        "Molecule ChEMBL ID",
        "Standard Value"
    ]
]

,Molecule ChEMBL ID,Standard Value
1296,CHEMBL267345,0.0


Number of compounds with  IC50 values less than or equal to zero.

In [12]:
(activities_nm["Standard Value"] <= 0).sum()

np.int64(1)

## Removal of Invalid Records and Recalculation of pIC50

Records with IC50 values less than or equal to zero were removed from the dataset. pIC50 values were then recalculated, and descriptive statistics were generated to verify the distribution of the cleaned activity data.

In [15]:
activities_nm = activities_nm[
    activities_nm["Standard Value"] > 0
].copy()
activities_nm["pIC50"] = (
    9 - np.log10(activities_nm["Standard Value"])
)
activities_nm["pIC50"].describe()

count    10256.000000
mean         4.950669
std          0.874793
min         -1.764200
25%          4.300000
50%          4.832240
75%          5.421361
max          9.920819
Name: pIC50, dtype: float64

## Identification of Extreme Activity Outliers

The dataset was sorted by pIC50 values to identify compounds with unusually low activity. This step was performed to detect potential outliers, data entry errors, or biologically insignificant compounds that could negatively impact downstream analyses and machine learning model performance.

In [ ]:
activities_nm.sort_values(
    "pIC50"
).head(20)[
    [
        "Molecule ChEMBL ID",
        "Standard Value",
        "pIC50"
    ]
]


,Molecule ChEMBL ID,Standard Value,pIC50
8826,CHEMBL523920,5.810319e+10,-1.764200
4992,CHEMBL4554875,1.970153e+10,-1.294500
8825,CHEMBL4534630,1.479108e+10,-1.170000
4876,CHEMBL197310,1.393157e+10,-1.144000
2281,CHEMBL4582361,9.210856e+09,-0.964300
4880,CHEMBL4435634,8.120822e+09,-0.909600
4991,CHEMBL4551488,4.380261e+09,-0.641500
10357,CHEMBL255758,1.000000e+08,1.000000
8315,CHEMBL830,1.000000e+07,2.000000
10952,CHEMBL2079699,7.089600e+06,2.149378


## Summary Statistics of IC50 Values

Descriptive statistics of the IC50 values were generated to examine the overall distribution, central tendency, and range of activity measurements. This helped identify potential outliers and assess the quality of the bioactivity data before further cleaning and analysis.

In [17]:
activities_nm["Standard Value"].describe()

count    1.025600e+04
mean     1.255711e+07
std      6.509464e+08
min      1.200000e-01
25%      3.790000e+03
50%      1.471500e+04
75%      5.011872e+04
max      5.810319e+10
Name: Standard Value, dtype: float64

## Quantifying Extreme Outliers

The number of compounds with pIC50 values below 2 was determined. These compounds represent extremely weak activity and were assessed as potential outliers prior to constructing the final cleaned dataset.

In [18]:
(activities_nm["pIC50"] < 2).sum()

np.int64(8)

## Removal of Extreme Activity Outliers

Compounds with pIC50 values below 2 were removed from the dataset. These records correspond to extremely weak activity and were considered outliers. The remaining dataset was retained for subsequent analysis and machine learning model development.

In [19]:
activities_nm = activities_nm[
    activities_nm["pIC50"] >= 2
].copy()

print(activities_nm.shape)

(10248, 49)


## Investigation of Duplicate Compounds

Multiple activity measurements may exist for the same compound due to testing in different assays or experimental studies. Duplicate compound records were examined prior to creating the final compound-level dataset and number of duplicates were obtained.

In [21]:
activities_nm["Molecule ChEMBL ID"].duplicated().sum()

np.int64(3916)

## Handling Duplicate Compounds

Duplicate records were grouped by Molecule ChEMBL ID to create a single entry per compound. The median pIC50 value was retained as the representative activity because it is less affected by outliers than the mean. The corresponding SMILES structure was also preserved for each compound.

In [22]:
activities_clean = (
    activities_nm
    .groupby("Molecule ChEMBL ID")
    .agg({
        "pIC50": "median",
        "Smiles": "first"
    })
    .reset_index()
)

## Final Cleaned Activity Dataset

After removing invalid records, filtering extreme outliers, and consolidating duplicate compounds, the final activity dataset contained 6,332 unique compounds. Each record consists of a Molecule ChEMBL ID, a corresponding SMILES structure, and a representative pIC50 value.

In [23]:
print(activities_clean.shape)

(6332, 3)


The initial 5 rows and all 3 columns of the new dataset.

In [24]:
activities_clean.head()

,Molecule ChEMBL ID,pIC50,Smiles
0,CHEMBL1000,3.940555,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1
1,CHEMBL100210,4.443697,CCCC[C@H]1C(=O)O[C@@H]2O[C@@]3(CC)CC[C@H]4[C@H...
2,CHEMBL100740,3.800519,C[C@@H]1CC[C@H]2[C@@H](C)C(=O)O[C@@H]3OC(C)(C)...
3,CHEMBL104,5.242633,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1
4,CHEMBL1043,3.656424,Nc1ccc(S(=O)(=O)c2ccc(N)cc2)cc1


The cleaned dataset was saved to the folder 'data_clean' as 'activities_clean.csv'

In [25]:
activities_clean.to_csv(
    "../data/processed/activities_clean.csv",
    index=False
)

In [26]:
activities_clean.columns

Index(['Molecule ChEMBL ID', 'pIC50', 'Smiles'], dtype='str')